The goal of this notebook is to model the topics of our dataset with BERTopic

In [ ]:
from bertopic import BERTopic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("works_abstracts.csv")
data = data[data["abstract"].notna()]
abstracts = data["abstract"].to_list()
dates = data["year"]

/tmp/ipython-input-87-2371968851.py:1: DtypeWarning:

Columns (92,93,94,95,96,97,98,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,

# Training

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer # for deleting the stop words
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

umap_model = UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

custom_stop_words = ["current"]
all_stop_words = list(ENGLISH_STOP_WORDS.union(custom_stop_words))
vectorizer_model = CountVectorizer(
    stop_words=all_stop_words,
    lowercase=True,
    token_pattern=r"(?u)\b\w\w+\b"
)

topic_modelling = BERTopic(embedding_model=embedding_model, umap_model=umap_model, language="english", vectorizer_model=vectorizer_model)
topics, probs = topic_modelling.fit_transform(abstracts)

In [ ]:
topic_modelling.save("bertopic_model")

2025-07-14 21:31:06,535 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [ ]:
data["topic"] = topics


In [ ]:
data.to_csv("data_topic.csv")

# Result analysis

In [ ]:
topic_modelling = BERTopic.load("bertopic_model")

In [ ]:
topic_modelling.visualize_topics()

In [ ]:
topic_modelling.visualize_barchart(top_n_topics=200)


In [ ]:
test = topic_modelling.get_topics()
test = pd.DataFrame(test)
test.to_csv("topic_words.csv")

In [ ]:
documents_annotes = topic_modelling.get_document_info(abstracts)
documents_annotes.to_csv("documents.csv")

coronavirus : topic 79

ecstasy, mdma : 74

nicotine : 73

cocaïne : 59 et 29

opioïdes : 56 et 34

asthma : 31

14 : diabete

5 : cancer

2 : dopamine

1 : canabis

In [ ]:
data = data[data["topic"].isin([74, 73, 59, 29, 56, 34])]
timestamps = pd.to_datetime(dates, format="%Y").to_list()  # liste des dates correspondantes
topics_over_time = topic_modelling.topics_over_time(
    docs = data["abstract"].to_list(),
    topics = data["topic"].to_list(),
    timestamps = pd.to_datetime(data["year"], format="%Y").to_list(),
    global_tuning=True,  # ou False selon la granularité voulue
    evolution_tuning=True,  # pour détecter des variations dans le vocabulaire
    nr_bins=20  # nombre de périodes de temps (buckets)
)
# convertir topics_over_time en df et filtrer un ou qq topics
topic_modelling.visualize_topics_over_time(topics_over_time)

In [ ]:
data = data[data["topic"].isin(range(0,6))]
timestamps = pd.to_datetime(dates, format="%Y").to_list()  # liste des dates correspondantes
topics_over_time = topic_modelling.topics_over_time(
    docs = data["abstract"].to_list(),
    topics = data["topic"].to_list(),
    timestamps = pd.to_datetime(data["year"], format="%Y").to_list(),
    global_tuning=True,  # ou False selon la granularité voulue
    evolution_tuning=True,  # pour détecter des variations dans le vocabulaire
    nr_bins=20  # nombre de périodes de temps (buckets)
)
# convertir topics_over_time en df et filtrer un ou qq topics
topic_modelling.visualize_topics_over_time(topics_over_time)

In [ ]:
# pd.crosstab pour analyse catégorielle (puis heatmap de la crosstab)

In [ ]:
# aide à réduire le nombre de topics
topic_modelling.visualize_hierarchy(top_n_topics=50)

In [ ]:
topic_modelling.visualize_heatmap(n_clusters=20, width=1000, height=1000)

In [ ]:
topic_modelling.reduce_topics(abstracts, nr_topics=10)

NLP:
- Tokenization
- Stop words
- Stemming/Lemmatization
- Bag-of-words, TF-IDF,...
- Embedding (SpaCy)